<div style="display:flex; justify-content:flex-end; align-items:center; gap:12px;">
    <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcT_QE36kR-JtQcZMteOxCgOglStNLOx8u1ufg&s" style="height:64px; width:auto;">
    <img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/logo-bdc.png" width="64">
</div>

# <span style="color:#336699">Acesso ao produto RTC Sentinel-1 pronto para uso do Brasil com Python e STAC</span>
<hr style="border:2px solid #0077b9;">

<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
    Carina Regina de Macedo <sup><a href="https://orcid.org/0000-0001-6732-9554"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>,Rennan Marujo<sup><a href="https://orcid.org/0000-0002-0082-9498"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>, Gilberto R. Queiroz<sup><a href="https://orcid.org/0000-0001-7534-0219"><i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brasil
    <br/><br/>
    Contact: <a href="mailto:brazildatacube@inpe.br">brazildatacube@inpe.br</a>
    <br/><br/>
    Última atualização: 29 de Maio de 2026
</div>

<br/>

<div style="text-align: justify;  margin-left: 15%; margin-right: 15%;">
<b>Abstract.</b> > Este Jupyter Notebook apresenta um fluxo de trabalho para acesso, visualização e classificação de imagens SAR do Sentinel-1 utilizando Python e bibliotecas geoespaciais. É demonstrado exemplo de acesso a dados raster Sentinel-1 RTC em formato GeoTIFF por meio de um serviço STAC, além de recortes espaciais a partir de coordenadas geográficas, conversão dos valores de retroespalhamento para escala logarítmica em decibéis (dB) e visualização de imagens SAR em ambiente Python. O material inclui análise estatística da distribuição dos valores de retroespalhamento SAR associados às classes água e terra por meio de histogramas. A partir dessas distribuições, é aplicado um procedimento de thresholding para segmentação de áreas alagadas e corpos d'água em imagens SAR, resultando na geração de máscaras binárias de detecção de água. O notebook foi desenvolvido como material de apoio à palestra <em>“Sentinel-1 com Correção Radiométrica do Terreno: acesso e uso dos dados com Python”</em>, voltada a atividades introdutórias em sensoriamento remoto por radar, processamento de imagens SAR e classificação baseada em limiares. O caso de estudo adotado corresponde à enchente histórica de 2024 no Rio Grande do Sul.
</div>

<br/>
<div style="text-align: justify;  margin-left: 25%; margin-right: 25%;font-size: 75%; border-style: solid; border-color: #0077b9; border-width: 1px; padding: 5px;">
    <b>For an introduction to the SpatioTemporal Asset Catalog (STAC) with the <em>Brazil Data Cube</em> infrastructure, please, refer to the following Jupyter Notebook:</b>
    <div style="margin-left: 10px; margin-right: 10px">
    Zaglia, M.; Marujo, R.; Queiroz, G. R.; Carlos, F. M. <a href="./stac-introduction.ipynb" target="_blank">Introduction to the SpatioTemporal Asset Catalog (STAC)</a>.
    </div>
</div>

<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/stac/stac.png?raw=true" align="right" width="66"/>

# Instalar as bibliotecas necessárias
<hr style="border:1px solid #0077b9;">

Para executar os exemplos apresentados neste Jupyter Notebook, é necessário instalar o cliente STAC para Python, que será utilizado para acessar os produtos RTC do Sentinel-1 disponibilizados na Base de Informações Georreferenciadas (BIG/INPE). Além disso, serão utilizadas as bibliotecas rasterio, numpy, matplotlib, geopandas e os, responsáveis pelas etapas de leitura, processamento, visualização e manipulação dos dados geoespaciais.

In [ ]:
#!pip install pystac-client rasterio numpy matplotlib geopandas

<img src="https://raw.githubusercontent.com/brazil-data-cube/code-gallery/master/img/stac/stac.png?raw=true" align="right" width="66"/>

# API Cliente STAC
<hr style="border:1px solid #0077b9;">

Para executar os exemplos apresentados neste Jupyter Notebook e acessar as funcionalidades da API cliente STAC será necessário importar o pacote `pystac_client`, conforme apresentado a seguir:

In [ ]:
import pystac_client

Em seguida, você pode verificar a versão instalada do pacote `pystac_client`:

In [ ]:
pystac_client.__version__

Crie um objeto pystac_client.Client conectado ao serviço STAC da BIG/INPE:

In [ ]:
service = pystac_client.Client.open('https://data.inpe.br/bdc/stac/v1/')

# Buscar pelos dados RTC Sentinel-1
<hr style="border:1px solid #0077b9;">

Neste exemplo, utilizaremos a API search do STAC para recuperar imagens SAR RTC Sentinel-1, a partir da coleção 'Sentinel-1 - Level-2 - Radiometrically Terrain Corrected (RTC)'. Esses dados consistem em valores de retroespalhamento SAR (γ⁰,gamma-0) normalizados em relação à topografia, derivados dos dados GRD. Cada item STAC representa uma cena SAR individual, contendo dados de retroespalhamento em diferentes polarizações (por exemplo, VV e VH), além de metadados associados à aquisição. A busca será realizada utilizando um retângulo envolvente (bounding box) definido pelos seguintes limites: $x_{min} = -51.57$, $x_{max} = -50.95$, $y_{min} = -30.17$, $y_{max} = -29.69$, correspondente a uma área que abrange Porto Alegre (Rio Grande do Sul) e seu entorno. Como os dados SAR são adquiridos ao longo do tempo, será aplicado um filtro temporal para selecionar imagens dentro de dois períodos de interesse (antes e depois da enchente histórica de 2024).  

In [ ]:
# O parâmetro 'bbox' define a área de interesse (bounding box),
lon_min = -51.57
lon_max = -50.95
lat_min = -30.17
lat_max = -29.69
bbox = (lon_min,lat_min,lon_max,lat_max)

In [ ]:
# Realiza a busca no catálogo STAC do INPE pela coleção 'sentinel-1-grd-bundle-1'.
# retornando apenas os tiles TOPODATA que intersectam essa região.
datas = ["2023-11-25", "2024-05-08"]

item_search = service.search(
    collections=['sentinel-1-rtc-1'],
    bbox=bbox,
    datetime=datas
)

A consulta acima deve retornar 2 itens STAC correspondentes aos produtos RTC Sentinel-1 que intersectam a extensão espacial solicitada.

In [ ]:
print(f"Número total de imagens: {item_search.matched()}")
items = list(item_search.items())
print(f"Lista de imagens: {items}")
print(f"Assets do item: {items[0].assets.keys()}")

# Carregar bibliotecas Python
<hr style="border:1px solid #0077b9;">

Uma vez que o dado RTC é recuperados via STAC, a execução do fluxo de trabalho requer a importação das bibliotecas rasterio, numpy, matplotlib.pyplot e geopandas, conforme apresentado a seguir:

In [ ]:
import rasterio
from rasterio.transform import rowcol
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import geopandas as gpd

# Recorte Espacial 
<hr style="border:1px solid #0077b9;">

O dado RTC pode abranger áreas além da região de interesse. Por isso, realiza-se o recorte do mosaico com base no limite de uma área de interesse definida definida acima:

In [ ]:
# Converte o resultado da busca em uma lista
items = list(item_search.items())

# Lista para armazenar as imagens e seus metadados
imagens = []

# Percorre todos os itens retornados pela busca
for i, item in enumerate(items, start=1):

    print(f"\nProcessando imagem {i}")
    print(f"ID: {item.id}")
    print(f"Data: {item.datetime}")

    # URL do asset Gamma0_VV
    file_path = item.assets["Gamma0_VV"].href

    print(file_path)

    # Abre o raster
    with rasterio.open(file_path) as src:

        # Recorte geográfico
        window = src.window(
            lon_min,
            lat_min,
            lon_max,
            lat_max
        )

        # Lê a imagem recortada
        img_ = src.read(1, window=window).astype(float)

        # Transformação espacial do recorte
        transform = src.window_transform(window)

        # Offsets do recorte na imagem original
        row_offset = int(window.row_off)
        col_offset = int(window.col_off)

        # Sistema de referência de coordenadas (CRS)
        raster_crs = src.crs

    # Armazena a imagem e seus metadados em uma lista
    imagens.append({
    "id": item.id,
    "data": item.datetime,
    "img": img_,
    "transform": transform,
    "row_offset": row_offset,
    "col_offset": col_offset,
    "crs": raster_crs
    })

# Exibe um resumo das imagens carregadas
print(f"\nTotal de imagens carregadas: {len(imagens)}")


# Conversão dos valores de retroespalhamento para decibéis (dB)  
<hr style="border:1px solid #0077b9;">

Os dados RTC são originalmente fornecidos em escala linear de retroespalhamento. Para facilitar a interpretação e a análise dos valores, esses dados são convertidos para a escala logarítmica em decibéis (dB). Essa transformação melhora a visualização das feições na imagem e torna mais evidente o contraste entre diferentes tipos de cobertura do terreno.

In [ ]:
for imagem in imagens:
    imagem["img"] = np.where(imagem["img"] > 0, imagem["img"], np.nan)
    imagem["img_db"] = 10 * np.log10(imagem["img"])

Após a leitura, o recorte das imagens e sua conversão para decibéis, o próximo passo é a visualização dos dados. Neste exemplo, os valores de retroespalhamento (backscatter) são exibidos em decibéis (dB) utilizando uma escala de cinza. Os parâmetros vmin e vmax definem o intervalo de valores exibidos na imagem, permitindo destacar os alvos de interesse e melhorar o contraste visual. Além disso, o parâmetro extent é utilizado para que os eixos do gráfico representem as coordenadas geográficas (longitude e latitude) da área recortada, enquanto a barra de cores indica os valores de retroespalhamento correspondentes a cada tonalidade da imagem.

In [ ]:
vmin = -20
vmax = 0

# Cria uma figura com duas colunas
fig, axes = plt.subplots(
    1, 2,
    figsize=(16, 8),
    constrained_layout=True
)

# Percorre as duas imagens
for ax, imagem in zip(axes, imagens[::-1]):

    im = ax.imshow(
        imagem["img_db"],
        cmap="gray",
        vmin=vmin,
        vmax=vmax,
        extent=[
            lon_min,
            lon_max,
            lat_min,
            lat_max
        ],
        origin="upper"
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    # Exibe a data da imagem no título
    ax.set_title(
        f"Sentinel-1 SAR\n{imagem['data'].strftime('%Y-%m-%d')}"
    )

# Barra de cores única para as duas imagens
fig.colorbar(
    im,
    ax=axes,
    label="Backscatter (dB)",
    shrink=0.8
)

plt.show()

Para analisar a distribuição dos valores de retroespalhamento SAR das classes água e terra, foram selecionados 30 pontos amostrais representativos de cada classe, utilizando como referência a imagem adquirida antes da ocorrência da enchente histórica. Esses pontos, armazenados em arquivos GeoPackage, são carregados no ambiente de trabalho e reprojetados para o mesmo sistema de referência de coordenadas da imagem RTC, garantindo a correta correspondência espacial entre as amostras e os pixels da imagem. Posteriormente, os valores de retroespalhamento serão extraídos nesses locais e utilizados na construção de histogramas para comparar o comportamento estatístico das duas classes.

In [ ]:
agua_points = gpd.read_file(
    "/home/jovyan/Documents/python_codes/Big_Tech_Talks/agua_amostras.gpkg" 
)

terra_points = gpd.read_file(
    r"/home/jovyan/Documents/python_codes/Big_Tech_Talks/terra_amostras.gpkg"
)

# garantir mesmo CRS
agua_points = agua_points.to_crs(raster_crs)
terra_points = terra_points.to_crs(raster_crs)

Os pontos representativos das classes água e terra são sobrepostos à imagem RTC, permitindo uma inspeção visual da sua distribuição espacial.

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

ax.imshow(
    imagens[1]["img_db"],
    cmap="gray",
    vmin=vmin,
    vmax=vmax,
    extent=[
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ],
    origin="upper"
)

# água
agua_points.plot(
    ax=ax,
    color="cyan",
    edgecolor="black",
    markersize=10,
    label="Água"
)

# terra
terra_points.plot(
    ax=ax,
    color="red",
    edgecolor="black",
    markersize=10,
    label="Terra"
)

ax.legend()

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.set_title("Sentinel-1 + Amostras")

plt.show()

Os valores de retroespalhamento correspondentes às classes "água" e "terra" são extraídos:

In [ ]:
def extrair_amostras(img, transform, pontos):

    valores = []

    for geom in pontos.geometry:

        row, col = rowcol(transform, geom.x, geom.y)

        if 0 <= row < img.shape[0] and 0 <= col < img.shape[1]:
            valores.append(img[row, col])

    return np.array(valores)

imagem = imagens[1]

agua_vals = extrair_amostras(
    imagem["img_db"],
    imagem["transform"],
    agua_points
)

terra_vals = extrair_amostras(
    imagem["img_db"],
    imagem["transform"],
    terra_points
)

Por fim, os valores de retroespalhamento das classes água e terra são representados por meio de histogramas. A comparação entre as distribuições permite visualizar o comportamento estatístico de cada classe e identificar a faixa de valores em que ocorre sua separação. Com base nessa análise, é possível definir um limiar (threshold) de retroespalhamento para discriminar automaticamente áreas de água e terra na imagem SAR. Em geral, superfícies de água apresentam valores de retroespalhamento mais baixos devido à reflexão especular da energia eletromagnética, enquanto superfícies terrestres tendem a apresentar valores mais elevados em razão de sua maior rugosidade e dos diferentes mecanismos de espalhamento presentes.

In [ ]:
plt.figure(figsize=(8,6))

plt.hist(
    agua_vals,
    bins=20,
    alpha=0.6,
    label="Água"
)

plt.hist(
    terra_vals,
    bins=20,
    alpha=0.6,
    label="Terra"
)

plt.xlabel("Backscatter (dB)")
plt.ylabel("Frequência")

plt.title("Histograma SAR")

plt.legend()

plt.show()

Com base na análise dos histogramas, é definido um limiar (threshold) de retroespalhamento para separar automaticamente as classes água e terra. Neste exemplo, foi adotado um limiar de −14 dB, de modo que todos os pixels com valores inferiores a esse limite são classificados como água, originando uma máscara binária em que os pixels classificados como água assumem o valor True e os demais False. A partir das máscaras de água obtidas para as imagens pré e pós-evento de inundação, é possível identificar as áreas que foram efetivamente inundadas entre as duas datas. Para isso, aplica-se uma operação lógica entre as máscaras binárias, considerando como área inundada apenas os pixels que eram classificados como terra na imagem pré-evento e passaram a ser classificados como água na imagem pós-evento. Dessa forma, são descartadas as superfícies de água permanentes, destacando-se exclusivamente as novas áreas alagadas associadas ao evento de inundação. O resultado é uma máscara binária em que os pixels inundados assumem o valor True e os demais False. Essa máscara é então representada sobre a extensão geográfica da imagem, permitindo visualizar a distribuição espacial das áreas inundadas identificadas entre as duas datas.

In [ ]:
threshold = -13

# Máscaras de água
water_pre = imagens[1]["img_db"] < threshold
water_pos = imagens[0]["img_db"] < threshold

# Área inundada: era terra e virou água
flood = (~water_pre) & (water_pos)

from matplotlib.colors import ListedColormap

cmap = ListedColormap(["white", "blue"])

plt.figure(figsize=(10, 8))

plt.imshow(
    flood,
    cmap=cmap,
    extent=[
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ],
    origin="upper"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.title(
    f"Área inundada - análise preliminar\n"
    f"{imagens[0]['data'].strftime('%Y-%m-%d')} → "
    f"{imagens[1]['data'].strftime('%Y-%m-%d')}"
)

legend_elements = [
    Patch(facecolor="blue", edgecolor="blue", label="Área inundada"),
    Patch(facecolor="white", edgecolor="black", label="Não inundada")
]

plt.legend(
    handles=legend_elements,
    loc="lower right"
)

plt.savefig(
    r"/home/jovyan/Documents/python_codes/Big_Tech_Talks/flood_extent.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

É importante ressaltar que este resultado representa uma detecção preliminar das áreas inundadas e não um mapa final de inundação. Em aplicações operacionais, é comum realizar etapas adicionais de processamento para reduzir falsas detecções e aumentar a confiabilidade do produto. Essas etapas podem incluir a utilização de modelos digitais de elevação (DEM), do índice HAND (Height Above the Nearest Drainage), filtros espaciais para remoção de ruídos, informações sobre declividade e conectividade hidrológica, entre outros critérios que auxiliam correta classificação das classes área inundade e não inundada. 